# Pokemon card prices - every set

Pulls the price history of every card in every set from pricecharting.com.

- Sets and card lists both come from pricecharting, so the card URLs are always correct (no guessing slugs from names).
- Sets are pulled oldest first (Base Set through to the newest release), with promo sets last. Release dates come from each set's card pages and are saved to `data/set_release_dates.csv` so they're only looked up once.
- Each set is saved to `data/sets/<set>.csv` as soon as it finishes. Re-running the loop skips sets that are already saved, so it can be stopped and restarted.
- Requests are paced adaptively: they speed up towards `MIN_DELAY` while the site is happy and back off hard at the first sign of rate limiting. If it keeps getting limited the run stops rather than risk a ban.
- A full run still takes a long time. Use `LANGUAGES` / `SETS_TO_PULL` to narrow it down.

In [ ]:
# packages
import pandas as pd
import numpy as np
import matplotlib as mpl
from bs4 import BeautifulSoup
from pathlib import Path
import requests
import random
import json
import re
import time

In [6]:
# settings
BASE_URL = "https://www.pricecharting.com"
START_DELAY = 2               # seconds between requests to begin with
MIN_DELAY = 1.5               # never go faster than this (1s got the odd 429 in testing)
MAX_DELAY = 60                # never go slower than this
MAX_RETRIES = 5               # rate limited this many times in a row on one request -> stop the run
LANGUAGES = ["English"]       # any of "English", "Japanese", "Chinese", "Korean"
SETS_TO_PULL = None           # None for every set, or a list of slugs e.g. ["pokemon-base-set", "pokemon-jungle"]
DATA_DIR = Path("data/sets")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RELEASE_DATES_FILE = Path("data/set_release_dates.csv")

# release dates to use instead of the scraped ones, for sets where pricecharting's date is wrong
RELEASE_DATE_OVERRIDES = {
    "pokemon-go": "2022-07-01",
}

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

# pricecharting's chart keys -> the grade shown on the site
GRADES = {
    "used": "Ungraded",
    "cib": "Grade 7",
    "new": "Grade 8",
    "graded": "Grade 9",
    "boxonly": "Grade 9.5",
    "manualonly": "PSA 10",
}

In [7]:
# adaptive rate limiting
# speed up a little after every good response, and when the site pushes back
# (429 / 403 / 503, a Cloudflare challenge page or a dropped connection):
#   - double the delay
#   - raise the floor so we don't creep back to the speed that got us limited
#   - cool off before retrying, and stop the run if it keeps happening
BLOCK_STATUS = {403, 429, 503}
BLOCK_PAGE = re.compile(r"just a moment|cf-chl|captcha|access denied|too many requests", re.I)

class RateLimited(Exception):
    pass

delay = START_DELAY
floor = MIN_DELAY
last_request = 0.0

def polite_request(method, url, **kwargs):
    """Make a request, keeping requests `delay` apart and backing off when rate limited"""
    global delay, floor, last_request

    for attempt in range(1, MAX_RETRIES + 1):
        wait = delay * random.uniform(0.8, 1.2) - (time.time() - last_request)
        if wait > 0:
            time.sleep(wait)
        last_request = time.time()

        try:
            res = session.request(method, url, timeout=30, **kwargs)
            limited = res.status_code in BLOCK_STATUS or bool(BLOCK_PAGE.search(res.text))
        except (requests.ConnectionError, requests.Timeout):
            res, limited = None, True

        if not limited:
            res.raise_for_status()
            delay = max(floor, delay * 0.95)
            return res

        floor = min(MAX_DELAY, delay * 1.5)
        delay = min(MAX_DELAY, delay * 2)

        retry_after = res.headers.get("Retry-After", "") if res is not None else ""
        cooldown = int(retry_after) if retry_after.isdigit() else 60 * attempt
        print(f"Rate limited ({res.status_code if res is not None else 'connection error'}), "
              f"cooling off {cooldown}s, delay now {delay:.1f}s (floor {floor:.1f}s)")
        time.sleep(cooldown)

    raise RateLimited(f"still rate limited after {MAX_RETRIES} tries on {url}")

In [8]:
# pull all set names
res = polite_request("GET", f"{BASE_URL}/category/pokemon-cards")
soup = BeautifulSoup(res.text, "html.parser")

card_sets = {}

for link in soup.select('a[href^="/console/pokemon"]'):
    card_sets[link["href"].split("/")[-1]] = link.get_text(strip=True)

sets_df = pd.DataFrame(card_sets.items(), columns=["set_slug", "set_name"])
sets_df["language"] = sets_df["set_name"].str.extract(r"(Japanese|Chinese|Korean)", expand=False).fillna("English")

sets_df

,set_slug,set_name,language
0,pokemon-30th-celebration,Pokemon 30th Celebration,English
1,pokemon-promo,Pokemon Promo,English
2,pokemon-base-set,Pokemon Base Set,English
3,pokemon-ascended-heroes,Pokemon Ascended Heroes,English
4,pokemon-pitch-black,Pokemon Pitch Black,English
...,...,...,...
296,pokemon-unleashed,Pokemon Unleashed,English
297,pokemon-unseen-forces,Pokemon Unseen Forces,English
298,pokemon-vivid-voltage,Pokemon Vivid Voltage,English
299,pokemon-white-flare,Pokemon White Flare,English


In [9]:
def get_set_cards(set_slug):
    """List every card in a set (all variants) from its pricecharting page"""
    url = f"{BASE_URL}/console/{set_slug}"
    rows = []
    cursor = 0

    # the page shows 150 products at a time, the "more results" form gives the next cursor
    while cursor is not None:
        res = polite_request("POST", url, data={"sort": "", "when": "none", "cursor": cursor})
        soup = BeautifulSoup(res.text, "html.parser")

        for tr in soup.select("table#games_table tbody tr[data-product]"):
            link = tr.select_one("td.title a")
            rows.append((tr["data-product"], link.get_text(" ", strip=True), link["href"]))

        next_page = soup.select_one("form.js-next-page input[name=cursor]")
        cursor = next_page["value"] if next_page else None

    cards = pd.DataFrame(rows, columns=["product_id", "title", "url"])

    # keep cards only (sealed product like boosters / blisters has no #number)
    cards["number"] = cards["title"].str.extract(r"#\s*(\S+)$", expand=False)
    cards = cards.dropna(subset=["number"]).reset_index(drop=True)

    cards["name"] = cards["title"].str.extract(r"^(.*?)\s*(?:\[|#)", expand=False).str.strip().str.lower()
    cards["variant"] = cards["title"].str.extract(r"\[(.*?)\]", expand=False)
    cards["url"] = BASE_URL + cards["url"].str.replace(BASE_URL, "", regex=False)
    cards["set_slug"] = set_slug

    return cards

base_set_cards = get_set_cards("pokemon-base-set")
base_set_cards

,product_id,title,url,number,name,variant,set_slug
0,715593,Charizard [1st Edition] #4,https://www.pricecharting.com/game/pokemon-bas...,4,charizard,1st Edition,pokemon-base-set
1,630417,Charizard #4,https://www.pricecharting.com/game/pokemon-bas...,4,charizard,NaN,pokemon-base-set
2,715597,Machamp [1st Edition] #8,https://www.pricecharting.com/game/pokemon-bas...,8,machamp,1st Edition,pokemon-base-set
3,630423,Mewtwo #10,https://www.pricecharting.com/game/pokemon-bas...,10,mewtwo,NaN,pokemon-base-set
4,630415,Blastoise #2,https://www.pricecharting.com/game/pokemon-bas...,2,blastoise,NaN,pokemon-base-set
...,...,...,...,...,...,...,...
443,9138757,Super Energy Removal [1999-2000] #79,https://www.pricecharting.com/game/pokemon-bas...,79,super energy removal,1999-2000,pokemon-base-set
444,8243947,PlusPower [1999-2000] #84,https://www.pricecharting.com/game/pokemon-bas...,84,pluspower,1999-2000,pokemon-base-set
445,8469898,Item Finder [1999-2000] #74,https://www.pricecharting.com/game/pokemon-bas...,74,item finder,1999-2000,pokemon-base-set
446,9138758,Full Heal [1999-2000] #82,https://www.pricecharting.com/game/pokemon-bas...,82,full heal,1999-2000,pokemon-base-set


In [10]:
def poke_pull(card):
    """Pull a Pokemon cards value history from pricecharting.com"""
    html = polite_request("GET", card["url"]).text

    match = re.search(r"VGPC\.chart_data\s*=\s*(\{.*?\});", html, re.S)
    if match is None:
        raise ValueError("no chart data on page")

    chart_data = json.loads(match.group(1))

    df = pd.DataFrame(
        [(grade, ts, price) for grade, points in chart_data.items() for ts, price, in points],
        columns =["grade", "date", "price"]
    )

    df["grade"] = df["grade"].map(GRADES).fillna(df["grade"])
    df["date"] = pd.to_datetime(df["date"], unit="ms")
    df["price"] = df["price"] / 100

    for col in ["set_slug", "product_id", "name", "variant", "number"]:
        df[col] = card[col]

    return df

print(poke_pull(base_set_cards.iloc[0]))

         grade                date     price          set_slug product_id  \
0    Grade 9.5 2020-12-01 07:00:00      0.00  pokemon-base-set     715593   
1    Grade 9.5 2021-01-01 07:00:00      0.00  pokemon-base-set     715593   
2    Grade 9.5 2021-02-01 07:00:00  35794.50  pokemon-base-set     715593   
3    Grade 9.5 2021-03-01 07:00:00  57789.00  pokemon-base-set     715593   
4    Grade 9.5 2021-04-01 06:00:00  57789.00  pokemon-base-set     715593   
..         ...                 ...       ...               ...        ...   
421   Ungraded 2026-06-01 06:00:00   7495.70  pokemon-base-set     715593   
422   Ungraded 2026-07-01 06:00:00   3049.73  pokemon-base-set     715593   
423   Ungraded 2026-08-01 06:00:00   6650.00  pokemon-base-set     715593   
424   Ungraded 2026-09-01 06:00:00   9541.19  pokemon-base-set     715593   
425   Ungraded 2026-10-01 06:00:00   6472.44  pokemon-base-set     715593   

          name      variant number  
0    charizard  1st Edition      4  
1

In [11]:
def pull_set(set_slug):
    """Pull every card in a set and save it to DATA_DIR. Returns the list of cards that failed"""
    cards = get_set_cards(set_slug)
    frames = []
    failed = []

    for _, card in cards.iterrows():
        try:
            frames.append(poke_pull(card))
            print(f"Successfully added {set_slug} #{card['number']} - {card['title']} (delay {delay:.1f}s)")
        except RateLimited:
            raise
        except Exception as e:
            print(f"Error occured on {card['url']}: {e}")
            failed.append(card["url"])

    if frames:
        pd.concat(frames, ignore_index=True).to_csv(DATA_DIR / f"{set_slug}.csv", index=False)

    return failed

In [15]:
def get_release_date(set_slug, n_cards=3):
    """A set's release date: the median date shown on its first few card pages, so one wrong card can't move the set"""
    dates = []
    for url in get_set_cards(set_slug)["url"][:n_cards]:
        soup = BeautifulSoup(polite_request("GET", url).text, "html.parser")
        for td in soup.select("td.title"):
            if "Release Date" in td.get_text():
                dates.append(pd.to_datetime(td.find_next_sibling("td").get_text(strip=True), errors="coerce"))

    return pd.Series(dates, dtype="datetime64[ns]").median()


# order the sets: base set through to the newest, then promo sets last
to_pull = sets_df[sets_df["language"].isin(LANGUAGES)]
if SETS_TO_PULL is not None:
    to_pull = to_pull[to_pull["set_slug"].isin(SETS_TO_PULL)]

# release dates are saved as they're found, so each set is only looked up once
release_dates = {}
if RELEASE_DATES_FILE.exists():
    saved = pd.read_csv(RELEASE_DATES_FILE, parse_dates=["release_date"])
    release_dates = dict(zip(saved["set_slug"], saved["release_date"]))

missing = [s for s in to_pull["set_slug"] if s not in release_dates and s not in RELEASE_DATE_OVERRIDES]
for i, set_slug in enumerate(missing, start=1):
    print(f"[{i}/{len(missing)}] Getting release date for {set_slug}")
    try:
        release_dates[set_slug] = get_release_date(set_slug)
    except RateLimited:
        raise
    except Exception as e:
        print(f"Error occured on set {set_slug}: {e}")
        release_dates[set_slug] = pd.NaT

    pd.DataFrame(release_dates.items(), columns=["set_slug", "release_date"]).to_csv(RELEASE_DATES_FILE, index=False)

release_dates.update({slug: pd.Timestamp(date) for slug, date in RELEASE_DATE_OVERRIDES.items()})

sets_df["release_date"] = pd.to_datetime(sets_df["set_slug"].map(release_dates))
# some sets have a placeholder date (e.g. 1990-01-01), anything before the TCG existed is treated as unknown
sets_df["release_date"] = sets_df["release_date"].where(sets_df["release_date"] >= "1996-01-01")
sets_df["promo"] = sets_df["set_name"].str.contains("promo", case=False)

# sets with no release date go at the end of their group
to_pull = (
    sets_df[sets_df["set_slug"].isin(to_pull["set_slug"])]
    .sort_values(["promo", "release_date", "set_name"], na_position="last")
    .reset_index(drop=True)
)

to_pull

sets_df.to_csv("data/sets_info.csv", index=False)

In [13]:
# loop over every set in order, skipping sets that are already saved
failed_cards = {}

for i, set_slug in enumerate(to_pull["set_slug"], start=1):
    if (DATA_DIR / f"{set_slug}.csv").exists():
        print(f"[{i}/{len(to_pull)}] Skipping {set_slug}, already saved")
        continue

    print(f"[{i}/{len(to_pull)}] Pulling {set_slug}")
    try:
        failed = pull_set(set_slug)
        if failed:
            failed_cards[set_slug] = failed
    except RateLimited as e:
        # stop hitting the site, the half-done set isn't saved so it'll be redone next run
        print(f"Stopping: {e}. Wait a while (ideally hours) then re-run this cell.")
        break
    except Exception as e:
        print(f"Error occured on set {set_slug}: {e}")

failed_cards

[1/158] Skipping pokemon-base-set, already saved
[2/158] Skipping pokemon-1999-topps-movie, already saved
[3/158] Skipping pokemon-1999-topps-tv, already saved
[4/158] Skipping pokemon-burger-king, already saved
[5/158] Skipping pokemon-jungle, already saved
[6/158] Skipping pokemon-fossil, already saved
[7/158] Skipping pokemon-base-set-2, already saved
[8/158] Skipping pokemon-team-rocket, already saved
[9/158] Skipping pokemon-2000-topps-chrome, already saved
[10/158] Skipping pokemon-2000-topps-tv, already saved
[11/158] Skipping pokemon-gym-heroes, already saved
[12/158] Skipping pokemon-gym-challenge, already saved
[13/158] Skipping pokemon-neo-genesis, already saved
[14/158] Skipping pokemon-neo-discovery, already saved
[15/158] Skipping pokemon-southern-islands, already saved
[16/158] Skipping pokemon-neo-revelation, already saved
[17/158] Skipping pokemon-neo-destiny, already saved
[18/158] Skipping pokemon-legendary-collection, already saved
[19/158] Skipping pokemon-expediti

{}

In [14]:
# combine every saved set
pokemon_all = pd.concat(
    [pd.read_csv(path, dtype={"number": str, "product_id": str, "variant": str}, parse_dates=["date"]) for path in DATA_DIR.glob("*.csv")],
    ignore_index=True
)

pokemon_all = pokemon_all.merge(sets_df, on="set_slug", how="left")

print(pokemon_all.dtypes)
print(pokemon_all.info())

pokemon_all

grade                      str
date            datetime64[us]
price                  float64
set_slug                   str
product_id                 str
name                       str
variant                    str
number                     str
set_name                   str
language                   str
release_date    datetime64[us]
promo                     bool
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 13124130 entries, 0 to 13124129
Data columns (total 12 columns):
 #   Column        Dtype         
---  ------        -----         
 0   grade         str           
 1   date          datetime64[us]
 2   price         float64       
 3   set_slug      str           
 4   product_id    str           
 5   name          str           
 6   variant       str           
 7   number        str           
 8   set_name      str           
 9   language      str           
 10  release_date  datetime64[us]
 11  promo         bool          
dtypes: bool(1), datetime64[us](2)

,grade,date,price,set_slug,product_id,name,variant,number,set_name,language,release_date,promo
0,Grade 9.5,2022-06-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
1,Grade 9.5,2022-07-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
2,Grade 9.5,2022-08-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
3,Grade 9.5,2022-09-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
4,Grade 9.5,2022-10-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13124125,Ungraded,2026-06-01 06:00:00,0.52,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
13124126,Ungraded,2026-07-01 06:00:00,0.70,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
13124127,Ungraded,2026-08-01 06:00:00,0.66,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
13124128,Ungraded,2026-09-01 06:00:00,0.64,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
